# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5001 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [2]:
import functools

import datetime, time
from pathlib import Path
from functools import wraps
from time import perf_counter
from typing import Any, Callable


def show_list_length(func: Callable[..., Any]) -> Callable[..., Any]:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        for idx, arg in enumerate(args):
            if isinstance(arg, list):
                print(f"[show_list_length] args[{idx}] {len(arg)}")

        for key, value in kwargs.items():
            if isinstance(value, list):
                print(f"[show_list_length] kwargs['{key}'] {len(value)}")

        return func(*args, **kwargs)

    return wrapper


@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")


process_data([1, 1, [2, 3]], "xyz")

[show_list_length] args[0] 3
Przetwarzanie xyz


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [3]:
import functools
from datetime import datetime
import time


def logger(log_filename: str) -> Callable[..., Any]:
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            start = perf_counter()
            try:
                return func(*args, **kwargs)
            finally:
                duration = perf_counter() - start
                timestamp = datetime.now()
                with Path(log_filename).open("a", encoding="utf-8") as log_file:
                    log_file.write(
                        f"function={func.__name__}; date={timestamp}; "
                        f"duration_s={duration:.6f}\n"
                    )
        return wrapper
    return decorator


@logger('app.log')
@show_list_length
def demo_func(*args: Any, **kwargs: Any) -> None:
    pass


@logger('app.log')
def long_foo(*args: Any, **kwargs: Any) -> None:
    time.sleep(5)


long_foo()
demo_func(["a", "b"], 2.3, "str", [1, 2, 3, 4], x=[0, 0, 0], values=["a", "b", "c"], obj=[print, list])

[show_list_length] args[0] 2
[show_list_length] args[3] 4
[show_list_length] kwargs['x'] 3
[show_list_length] kwargs['values'] 3
[show_list_length] kwargs['obj'] 2


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [4]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl
Błędny format adresu email: invalid_at_email


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [5]:
class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.name)
        print(f"[LOG] Odczyt atrybutu '{self.name}': {value}")
        return value

    def __set__(self, instance, value):
        print(f"[LOG] Zapis atrybutu '{self.name}' = {value}")
        instance.__dict__[self.name] = value


class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek


u = Uzytkownik("Anna", 25)

print(u.imie)   # log odczytu
print(u.wiek)   # log odczytu

u.imie = "Katarzyna"   # log zapisu
u.wiek = 30            # log zapisu

print(u.imie)
print(u.wiek)

[LOG] Zapis atrybutu 'imie' = Anna
[LOG] Zapis atrybutu 'wiek' = 25
[LOG] Odczyt atrybutu 'imie': Anna
Anna
[LOG] Odczyt atrybutu 'wiek': 25
25
[LOG] Zapis atrybutu 'imie' = Katarzyna
[LOG] Zapis atrybutu 'wiek' = 30
[LOG] Odczyt atrybutu 'imie': Katarzyna
Katarzyna
[LOG] Odczyt atrybutu 'wiek': 30
30


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [6]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [7]:
def collatz_generator(n):
    while n != 1:
        yield n
        n = n // 2 if n % 2 == 0 else 3 * n + 1
    yield 1


for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [8]:
from functools import wraps

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(
                    f"Required: {role}, current: {current_user.get('role')}"
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator


@require_role("superuser")
def admin_panel():
    return "admin panel"

print(admin_panel())


current_user = {"username": "jan", "role": "user"}

@require_role("superuser")
def remove_user():
    return "user removed"

try:
    print(remove_user())
except PermissionError as e:
    print(e)

admin panel
Required: superuser, current: user


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [9]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"'{self.name}' must be {self.expected_type.__name__} but was {type(value).__name__}"
            )
        instance.__dict__[self.name] = value


class Person:
    name = Typed(str)
    age = Typed(int)

    def __init__(self, name, age):
        self.name = name
        self.age = age


Person("Anna", 25)

try:
    Person(1, 2)
except TypeError as e:
    print(e)


'name' must be str but was int


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [10]:
def prime_generator():
    num = 2
    while True:
        is_prime = True
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            yield num
        num += 1


primes_ending_with_7 = (p for p in prime_generator() if p % 10 == 7)
for _ in range(10):
    print(next(primes_ending_with_7))

7
17
37
47
67
97
107
127
137
157
